Lab 03 — Fleet & pin honesty (M3).

Open-Meteo ≠ in-situ station; public AIS ≠ own-edge; warning ≠ in-situ.
SIM devices must never be claimed LIVE.

Run:  python labs/lab03_honesty_claims.py

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexar76/gaia-atlas-sensors-course/blob/main/notebooks/{})


In [ ]:
# Setup — run this cell once per session
# Core course only — no network hub deps
import os
import subprocess
import sys

REPO = "https://github.com/alexar76/gaia-atlas-sensors-course.git"
DEST = "/content/gaia-atlas-sensors-course"

if not os.path.isdir(DEST):
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, DEST], check=True)
os.chdir(DEST)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)
os.environ.setdefault("COURSE_LANG", "en")  # change to ru or es


In [ ]:
"""Lab 03 — Fleet & pin honesty (M3).

Open-Meteo ≠ in-situ station; public AIS ≠ own-edge; warning ≠ in-situ.
SIM devices must never be claimed LIVE.

Run:  python labs/lab03_honesty_claims.py
"""

from __future__ import annotations



from courselib.i18n import get_translator
from courselib.sensors import (
    HONESTY_TABLE,
    LIVE_WEATHER_DEVICE,
    SIM_WEATHER_DEVICE,
    classify_mode,
    fleet_devices,
    honesty_claims,
    try_live,
)


def main() -> None:
    t = get_translator()
    print(f"== {t('modules.m3.title')} ==")
    print(t("modules.m3.concept"))
    print()

    print("Honesty claims (teaching table):")
    for row in HONESTY_TABLE:
        mark = "✓" if row["verdict"] == "honest" else "✗"
        print(f"  {mark} [{row['verdict']}] {row['claim']}")
        if row["correction"]:
            print(f"      → {row['correction']}")

    assert honesty_claims("om-as-station")["verdict"] == "dishonest"
    assert honesty_claims("fail-loud")["verdict"] == "honest"
    assert honesty_claims("public-ais-as-edge")["verdict"] == "dishonest"
    print(f"\n{t('ui.verify')}: honesty_claims classifier — ok")

    devices = try_live(fleet_devices, label="gaia.fleet.status@v1")
    if devices is not None:
        by_id = {d.get("device_id"): d for d in devices}
        for device_id in (LIVE_WEATHER_DEVICE, SIM_WEATHER_DEVICE):
            d = by_id.get(device_id)
            if not d:
                print(f"  (fleet missing {device_id})")
                continue
            mode = classify_mode(d)
            print(f"  {device_id}: mode={mode} source={d.get('source')!r}")
            if device_id == SIM_WEATHER_DEVICE:
                assert mode == "SIM"
            if device_id == LIVE_WEATHER_DEVICE:
                assert mode == "LIVE"

    print(f"\n--- {t('exercises.heading')} ---")
    print("Run: python labs/run_exercises.py --module m3")

main()
